# Phase 147: CryptoBench 650M-Transfer: Score-Only Eval

Loads Phase146's 8 transfer-learned checkpoints, builds the ensemble,
tunes threshold on VAL only, evaluates on TEST -- directly comparable
to Phase47's from-scratch 650M ensemble (test AUPRC 0.3627, AUC 0.8873,
MCC 0.3986).

## Cell 1: Install dependencies

In [ ]:
!pip install -q torch-geometric==2.6.1

## Cell 2: Setup, load Phase 45's CryptoBench-650M blob (identical to Phase46/47/146)

In [ ]:
import json, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from torch_geometric.utils import softmax as pyg_softmax
from sklearn.metrics import (average_precision_score, roc_auc_score,
                              matthews_corrcoef, f1_score, accuracy_score)

def locate(f):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(f))
    if not c: raise FileNotFoundError(f)
    return max(c, key=lambda p: p.stat().st_size)

blob = torch.load(locate("phase45_cryptobench_graphs_esm650m.pt"), map_location="cpu", weights_only=False)
dataset = blob["dataset"]
N_PRE = int(blob["n_pre_topo"]); TOPO_DIM = int(blob["topo_dim"]); POS_WEIGHT = float(blob["pos_weight"])
splits = {s: [d for d in dataset if d.split == s] for s in ("train","val","test")}

GPU_OK=False
if torch.cuda.is_available():
    try:
        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")
    except RuntimeError as e: print("GPU unusable:", e)
DEVICE = torch.device("cuda" if GPU_OK else "cpu")
print(f"device {DEVICE} | node dim {dataset[0].x.size(1)} | pos_weight {POS_WEIGHT:.2f} | " +
      " | ".join(f"{k} {len(v)}" for k,v in splits.items()))

WORK = Path("/kaggle/working"); (WORK/"cache").mkdir(parents=True, exist_ok=True)

## Cell 3: Model (identical to every prior localization phase)

In [ ]:
class EGCL(nn.Module):
    def __init__(s,h,ea=2,act=nn.SiLU,uc=True):
        super().__init__(); s.uc=uc
        s.e=nn.Sequential(nn.Linear(2*h+1+ea,h),act(),nn.Linear(h,h),act())
        s.n=nn.Sequential(nn.Linear(2*h,h),act(),nn.Linear(h,h))
        s.a=nn.Sequential(nn.Linear(h,1),nn.Sigmoid())
        if uc:
            s.c=nn.Sequential(nn.Linear(h,h),act(),nn.Linear(h,1,bias=False))
            nn.init.xavier_uniform_(s.c[-1].weight,gain=0.001)
        s.norm=nn.LayerNorm(h)
    def forward(s,h,x,ei,ea):
        src,dst=ei; rel=x[src]-x[dst]; d2=(rel**2).sum(-1,keepdim=True)
        m=s.e(torch.cat([h[src],h[dst],d2,ea],-1)); m=m*s.a(m)
        agg=torch.zeros_like(h).index_add_(0,dst,m)
        deg=torch.zeros(h.size(0),1,device=h.device).index_add_(0,dst,torch.ones_like(d2)).clamp(min=1)
        h=s.norm(h+s.n(torch.cat([h,agg/deg],-1)))
        if s.uc:
            w=s.c(m)*rel/(d2.sqrt()+1.0); x=x+torch.zeros_like(x).index_add_(0,dst,w)/deg
        return h,x

class Fus(nn.Module):
    def __init__(s,h,t):
        super().__init__(); s.p=nn.Linear(t,h); s.g=nn.Sequential(nn.Linear(h+t,h),nn.Sigmoid())
    def forward(s,h,t): return h+s.g(torch.cat([h,t],-1))*s.p(t)

class LocalizingEGNN(nn.Module):
    """Same TopologicalEGNN as every prior phase, plus a second head that
    predicts one pocket-center point per chain via a batch-aware softmax over
    a dedicated per-residue localization logit -- decoupled from the
    classification probability on purpose (see markdown above)."""
    def __init__(s, n_pre, td, h=128, L=4, dr=0.1, use_loc_head=True):
        super().__init__(); s.n_pre=n_pre; s.use_loc_head=use_loc_head
        s.embed=nn.Sequential(nn.Linear(n_pre+td,h),nn.SiLU(),nn.Linear(h,h))
        s.ls=nn.ModuleList([EGCL(h,uc=(i<L-1)) for i in range(L)])
        s.fs=nn.ModuleList([Fus(h,td) for _ in range(L)])
        s.head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Dropout(dr),nn.Linear(h//2,1))
        if use_loc_head:
            s.loc_head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Linear(h//2,1))

    def forward(s, d):
        h=s.embed(d.x); pos=d.pos; t=d.x[:,s.n_pre:]
        for l,f in zip(s.ls,s.fs): h,pos=l(h,pos,d.edge_index,d.edge_attr); h=f(h,t)
        logits = s.head(h).squeeze(-1)
        pred_center = None
        if s.use_loc_head:
            batch = d.batch if hasattr(d, "batch") and d.batch is not None \
                    else torch.zeros(h.size(0), dtype=torch.long, device=h.device)
            loc_logit = s.loc_head(h).squeeze(-1)
            attn = pyg_softmax(loc_logit, batch)                    # sums to 1 per graph
            weighted = attn.unsqueeze(-1) * pos                     # (N,3)
            n_graphs = int(batch.max().item()) + 1
            pred_center = torch.zeros(n_graphs, 3, device=h.device).index_add_(0, batch, weighted)
        return logits, pred_center

## Cell 4: Load Phase 146's 8 transfer-learned checkpoints into an ensemble

In [ ]:
ensemble = []
for seed in range(8):
    m = LocalizingEGNN(N_PRE, TOPO_DIM, use_loc_head=True).to(DEVICE)
    state = torch.load(locate(f"cryptobench_esm650m_transfer_seed{seed}.pt"), map_location=DEVICE)
    m.load_state_dict(state)
    m.eval()
    ensemble.append(m)
print(f"loaded {len(ensemble)} 650M-transfer checkpoints into ensemble")

## Cell 5: Pick a decision threshold on VAL ONLY (never touch test for this), then evaluate on TEST

In [ ]:
@torch.no_grad()
def predict_ensemble(loader):
    """Per-chain: pooled per-residue probs (mean across seeds) + predicted center."""
    all_probs, all_y, all_dcc = [], [], []
    for b in loader:
        b = b.to(DEVICE)
        seed_logits, seed_centers = [], []
        for m in ensemble:
            lg, pc = m(b)
            seed_logits.append(torch.sigmoid(lg).cpu().numpy())
            seed_centers.append(pc.cpu().numpy())
        p = np.mean(seed_logits, axis=0)
        c = np.mean(seed_centers, axis=0)   # (n_graphs_in_batch, 3)
        all_probs.append(p); all_y.append(b.y.cpu().numpy())
        true_c = b.true_center.view(-1,3).cpu().numpy()
        d_ = np.linalg.norm(c - true_c, axis=1)
        all_dcc.extend(d_.tolist())
    return np.concatenate(all_probs), np.concatenate(all_y), np.array(all_dcc)

val_loader = DataLoader(splits["val"], batch_size=16)
test_loader = DataLoader(splits["test"], batch_size=16)

val_p, val_y, val_dcc = predict_ensemble(val_loader)
test_p, test_y, test_dcc = predict_ensemble(test_loader)

# threshold sweep on VAL only, optimizing MCC (matches the field's own
# reporting convention -- MCC is the metric CryptoBench itself highlights
# for class-imbalanced residue classification)
best_thresh, best_mcc = 0.5, -1.0
for thresh in np.arange(0.05, 0.96, 0.01):
    mcc = matthews_corrcoef(val_y, (val_p >= thresh).astype(int))
    if mcc > best_mcc: best_mcc, best_thresh = mcc, thresh
print(f"val-tuned decision threshold: {best_thresh:.2f} (val MCC {best_mcc:.4f})")
print(f"(CryptoBench's OWN paper used 0.75 for their own model -- not directly "
      f"transferable to a different model's calibration, reported for reference only)")

## Cell 6: Field-standard residue classification metrics on TEST (pooled)

In [ ]:
test_auc = roc_auc_score(test_y, test_p)
test_auprc = average_precision_score(test_y, test_p)
test_pred_val_thresh = (test_p >= best_thresh).astype(int)
test_mcc = matthews_corrcoef(test_y, test_pred_val_thresh)
test_f1 = f1_score(test_y, test_pred_val_thresh)
test_acc = accuracy_score(test_y, test_pred_val_thresh)

# also report at the paper's own 0.75 threshold, for direct reference
test_pred_075 = (test_p >= 0.75).astype(int)
mcc_075 = matthews_corrcoef(test_y, test_pred_075)
f1_075 = f1_score(test_y, test_pred_075)

LIT = {
    "PocketMiner (structure+MD)": {"auc": 0.76, "auprc": 0.19, "mcc": 0.22, "f1": 0.78},
    "P2Rank-apo": {"auc": 0.81, "auprc": 0.21, "mcc": None, "f1": 0.81},
    "CryptoBench ESM2-NN (paper SOTA)": {"auc": 0.87, "auprc": 0.40, "mcc": 0.42, "f1": None},
}

print(f"{'method':<38}{'AUC':>8}{'AUPRC':>8}{'MCC':>8}{'F1':>8}")
print("-"*70)
for name, m in LIT.items():
    fmt = lambda v: f"{v:.3f}" if v is not None else "  --  "
    print(f"{name+' (literature)':<38}{fmt(m['auc']):>8}{fmt(m['auprc']):>8}{fmt(m['mcc']):>8}{fmt(m['f1']):>8}")
print(f"{'OURS (val-tuned thresh)':<38}{test_auc:>8.3f}{test_auprc:>8.3f}{test_mcc:>8.3f}{test_f1:>8.3f}")
print(f"{'OURS (0.75 thresh, paper-matched)':<38}{'':>8}{'':>8}{mcc_075:>8.3f}{f1_075:>8.3f}")
print(f"\nn={len(test_y):,} test residues across {len(splits['test'])} chains, "
      f"{int(test_y.sum())} cryptic-pocket positives ({100*test_y.mean():.2f}%)")

## Cell 7: Our own localization-head metric (novel -- no prior baseline exists for this exact task)

In [ ]:
DCC_THRESH = 4.0
dcc_rate = float((test_dcc < DCC_THRESH).mean())
print(f"Localization-head accuracy (predicted pocket center within {DCC_THRESH:.0f}A of the "
      f"true cryptic-pocket centroid): {dcc_rate:.4f}  (n={len(test_dcc)})")
print(f"mean center error: {test_dcc.mean():.2f}A  median: {np.median(test_dcc):.2f}A")
print(f"\nNo existing cryptic-pocket method predicts a trained 3D localization point -- "
      f"PocketMiner/P2Rank-apo/CryptoBench's own baseline are all per-residue classifiers. "
      f"This number has no direct literature comparison; report it as a new capability, "
      f"not a head-to-head win/loss.")

json.dump({
    "classification": {"auc": test_auc, "auprc": test_auprc, "mcc": test_mcc, "f1": test_f1,
                        "mcc_at_075": mcc_075, "f1_at_075": f1_075, "val_tuned_threshold": float(best_thresh)},
    "localization": {"dcc_rate_4A": dcc_rate, "mean_error_A": float(test_dcc.mean()),
                      "median_error_A": float(np.median(test_dcc))},
    "n_test_residues": int(len(test_y)), "n_test_chains": len(splits["test"]),
    "n_test_positives": int(test_y.sum()),
    "literature": LIT,
}, open(WORK/"phase147_esm650m_transfer_summary.json", "w"), indent=2)
print("\nsaved -> phase147_esm650m_transfer_summary.json")